In [211]:
pip install pandas streamlit scikit-learn  streamlit

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [212]:
import ast
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer






In [213]:
movies=pd.read_csv('/home/vipin/own projects/rec/data/tmdb_5000_movies.csv')
credits=pd.read_csv('/home/vipin/own projects/rec/data/tmdb_5000_credits.csv')


In [214]:
movies = movies.merge(credits, left_on='original_title', right_on='title')
movies.drop(columns='original_title',inplace=True)


In [215]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='object')

In [216]:
movies = movies.rename(columns={'title_x': 'title'})

In [217]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [218]:
movies.isnull().sum()
movies.dropna(inplace=True)

In [219]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4544 entries, 0 to 4546
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4544 non-null   int64 
 1   title     4544 non-null   object
 2   overview  4544 non-null   object
 3   genres    4544 non-null   object
 4   keywords  4544 non-null   object
 5   cast      4544 non-null   object
 6   crew      4544 non-null   object
dtypes: int64(1), object(6)
memory usage: 284.0+ KB


In [220]:
print(movies['genres'].iloc[0])


[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]


In [221]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

movies['genres'] = movies['genres'].apply(convert)


In [222]:
movies['keywords'] = movies['keywords'].apply(convert)

In [223]:
#extract top 3 actors only to reduce noise
def get_cast(text):
    L = []
    for i in ast.literal_eval(text)[:3]:  # take only first 3
        L.append(i['name'])
    return L

movies['cast'] = movies['cast'].apply(get_cast)

In [224]:
def get_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

movies['crew'] = movies['crew'].apply(get_director)

In [225]:

movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [226]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [227]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

In [228]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

In [229]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],Following the death of District Attorney Harve...
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],"John Carter is a war-weary, former military ca..."


vectorization   

In [230]:

cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(movies['tags']).toarray()

In [231]:
similarity = cosine_similarity(vectors)

In [232]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

vectors = tfidf.fit_transform(movies['tags']).toarray()

In [233]:

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

vectors = tfidf.fit_transform(movies['tags']).toarray()

In [234]:
movies['title'] = movies['title'].str.lower()

In [235]:
def recommend(movie):
    movie = movie.lower()
    
    # find index
    index = movies[movies['title'] == movie].index[0]
    
    # similarity scores
    distances = similarity[index]
    
    # sort movies based on similarity
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    # get movie names
    recommended_movies = []
    for i in movies_list:
        recommended_movies.append(movies.iloc[i[0]].title)
    
    return recommended_movies

In [236]:
print(recommend("avatar"))

['aliens', 'moonraker', 'alien', 'alien³', 'silent running']


In [237]:
import pickle

pickle.dump(movies, open('movies.pkl','wb'))
pickle.dump(similarity, open('similarity.pkl','wb'))

In [240]:
import numpy as np

embeddings = pickle.load(open('embeddings.pkl', 'rb'))

def recommend(movie):
    movie = movie.lower()

    if movie not in movies['title_lower'].values:
        return []

    index = movies[movies['title_lower'] == movie].index[0]

    # Compute similarity manually
    scores = np.dot(embeddings, embeddings[index]) / (
        np.linalg.norm(embeddings, axis=1) * np.linalg.norm(embeddings[index])
    )

    similar_movies = sorted(
        list(enumerate(scores)),
        key=lambda x: x[1],
        reverse=True
    )[1:11]

    return [(movies.iloc[i[0]].title, i[1]) for i in similar_movies]

FileNotFoundError: [Errno 2] No such file or directory: 'embeddings.pkl'